# 01. BPE 런타임 하네스 기초

목표: Belief, Progress, Experience를 분리하고 `track`, `commit`, `recall`, `note`가 어떤 상태를 읽고 쓰는지 확인합니다. Python 표준 라이브러리만 필요하며 위에서 아래로 실행합니다. 이 코드는 논문 구현의 소규모 학습용 모형입니다.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List

@dataclass
class BPEState:
    # Belief는 현재 관찰로 바뀌는 사실입니다.
    belief: Dict[str, str] = field(default_factory=dict)
    # Progress는 이번 에피소드의 하위 목표 상태입니다.
    progress: Dict[str, str] = field(default_factory=dict)
    # Experience는 다음 에피소드에도 재사용할 지식입니다.
    experience: List[str] = field(default_factory=list)
    pending_notes: List[str] = field(default_factory=list)

class Harness:
    def __init__(self, state: BPEState):
        self.state = state
        self.calls = 0

    def track(self, key: str, value: str) -> str:
        self.calls += 1
        self.state.belief[key] = value
        return f'belief[{key}]={value}'

    def commit(self, subgoal: str, status: str) -> str:
        self.calls += 1
        self.state.progress[subgoal] = status
        return f'progress[{subgoal}]={status}'

    def recall(self, keyword: str) -> List[str]:
        self.calls += 1
        return [item for item in self.state.experience if keyword in item]

    def note(self, insight: str) -> str:
        self.calls += 1
        self.state.pending_notes.append(insight)
        return insight


## 작은 장기 과제

목표는 `주전자를 찾아 씻고 선반에 놓기`입니다. 관찰 사실과 하위 목표를 한 문자열에 섞지 않고 서로 다른 수명 주기에 기록합니다.

In [ ]:
state = BPEState(experience=[
    'kettle: 주방부터 탐색',
    'clean: 싱크대에서 물로 씻기',
])
harness = Harness(state)

print(harness.commit('주전자 찾기', '진행 중'))
print('회상:', harness.recall('kettle'))
print(harness.track('kettle_location', '주방 조리대'))
print(harness.commit('주전자 찾기', '완료'))
print(harness.commit('주전자 씻기', '완료'))
print(harness.track('target_shelf', '식기 선반'))
print(harness.commit('선반에 놓기', '완료'))
print('새 통찰:', harness.note('kettle: 조리대를 먼저 확인'))

print('\nBelief:', state.belief)
print('Progress:', state.progress)
print('Experience:', state.experience)
print('통합 대기:', state.pending_notes)
print('하네스 호출 수:', harness.calls)


## 에피소드 경계에서 통합

실행 중 생긴 메모를 즉시 장기 기억으로 신뢰하지 않습니다. 과제가 성공했을 때만 중복을 제거해 Experience로 승격합니다.

In [ ]:
def consolidate(state: BPEState, success: bool) -> None:
    if success:
        for note in state.pending_notes:
            if note not in state.experience:
                state.experience.append(note)
    state.pending_notes.clear()

success = all(status == '완료' for status in state.progress.values())
consolidate(state, success)
assert success
assert 'kettle: 조리대를 먼저 확인' in state.experience
print('통합된 Experience:', state.experience)
print('핵심 관찰: B/P는 에피소드 상태이고 E는 검증 후 재사용됩니다.')
